<a href="https://colab.research.google.com/github/AlifHammam/data-science-2026/blob/main/Pertemuan12_AlifHammamMultazam_240401010043.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 12 - Asosiasi Data & Sistem Rekomendasi Dasar: Market Basket Analysis

**Nama Lengkap:** Alif Hammam Multazam  
**NIM:** 240401010043  
**Kelas:** IF403

## 1. Import Library & Generate Dataset Transaksi

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity

print('Library berhasil diimport!')

In [ ]:
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

## 2. One-Hot Encoding Transaksi

In [ ]:
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

df.head()

## 3. Cari Frequent Itemset dengan Apriori

In [ ]:
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

## 4. Bentuk & Saring Aturan Asosiasi

In [ ]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# Aturan dengan Lift tertinggi umumnya melibatkan Roti -> Selai,
# yang masuk akal secara bisnis karena kedua produk memang sering dikonsumsi bersamaan.

## 5. Rekomender Sederhana dengan Content-Based Filtering

In [ ]:
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

## 6. Bandingkan Association Rules vs Content-Based Filtering

In [ ]:
produk_target = 'Roti'

rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('\nRekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

### Diskusi

- Association Rules merekomendasikan Selai untuk Roti karena kedua produk memang sering muncul bersama di data transaksi (pola perilaku pembelian).
- Content-Based Filtering merekomendasikan produk lain dalam kategori Bakery/serupa berdasarkan atribut kategori, tanpa memandang pola transaksi.
- Kedua pendekatan tidak selalu memberikan hasil yang sama persis, namun keduanya saling melengkapi: Association Rules unggul saat data transaksi melimpah, sedangkan Content-Based lebih baik untuk mengatasi cold start pada produk baru yang belum punya histori transaksi. Pendekatan Hybrid dapat menggabungkan keduanya untuk hasil rekomendasi yang lebih baik.

## Kesimpulan

Pada praktikum pertemuan kedua belas ini saya mempelajari Association Rule Mining menggunakan algoritma Apriori beserta tiga metrik kuncinya: Support, Confidence, dan Lift, serta pengenalan dasar Sistem Rekomendasi melalui pendekatan Collaborative Filtering dan Content-Based Filtering.

Melalui dataset transaksi sintetis, saya memahami cara mencari frequent itemset dengan library mlxtend, membentuk dan menyaring aturan asosiasi berdasarkan min_confidence dan min_lift, serta membangun rekomender sederhana berbasis kemiripan kategori produk menggunakan cosine similarity. Saya juga membandingkan hasil rekomendasi dari kedua pendekatan dan memahami kapan masing-masing lebih tepat digunakan.

Keterbatasan pada praktikum ini adalah dataset transaksi yang digunakan bersifat sintetis dan berskala kecil (50 transaksi, 10 produk), sehingga pola asosiasi yang ditemukan belum tentu merepresentasikan perilaku belanja yang kompleks di dunia nyata. Namun, praktikum ini memberikan pemahaman yang baik mengenai fondasi Market Basket Analysis dan Sistem Rekomendasi dasar sebagai materi penutup mata kuliah Data Science.